##导入相关包

In [ ]:
import pandas as pd
import numpy as np
import jieba  #jieba分词
import os   
from sklearn.feature_extraction.text import TfidfVectorizer #TF-IDF
from sklearn.model_selection import GridSearchCV            #网格搜索
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import confusion_matrix,classification_report #预测效果的评估
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif']=['SimHei']  #这两行用于plt图像显示中文，否则plt无法显示中文
plt.rcParams['axes.unicode_minus']=False

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

##簡轉繁

In [ ]:
!pip3 install opencc-python-reimplemented

#数据读取(簡->繁)

In [ ]:
# 使用pandas读取数据集
import pandas as pd
# train = pd.read_table('train.txt', sep='\t',header=None)  # 训练集 占RAM太多了，當TFIDF矩陣的時候：(news_tfidf.toarray())
train = pd.read_table('/content/drive/MyDrive/中文新闻文本标题分类Data/train.txt', sep='\t', header=None, nrows=20000)  # 训练集，仍占用太多，限制只讀20000筆
test = pd.read_table('/content/drive/MyDrive/中文新闻文本标题分类Data/test.txt', sep='\t',header=None, nrows=20000) # 测试集，仍占用太多，限制只讀20000筆

In [ ]:
# 添加列名便于对数据进行更好处理
train.columns = ["text_a",'label']
# dev.columns = ["text_a",'label']
test.columns = ["text_a"]

In [ ]:
# 定义要进行分类的14个类别
label_list=list(train.label.unique())
print(label_list)
print(len(label_list))  # 一定要14个类别

In [ ]:
# 查看训练集
train

In [ ]:
test

##遍歷 train 變數的每一行，然後將每一行的簡體字轉換為繁體字

In [ ]:
from opencc import OpenCC

cc = OpenCC('s2tw')
for index, row in train.iterrows():
  train.loc[index, 'label'] = cc.convert(str(row['label']))
  train.loc[index, 'text_a'] = cc.convert(row['text_a'])
for index, row in test.iterrows():
  #test.loc[index, 'label'] = cc.convert(str(row['label']))
  test.loc[index, 'text_a'] = cc.convert(row['text_a'])

In [ ]:
# 復原train_tmp用，便實驗
train_tmp=train.copy
# 查看训练集
train

##统计

In [ ]:
# 总类别标签分布统计
train['label'].value_counts()

In [ ]:
# 文本长度统计分析,通过分析可以看出文本较短，最长为48
train['text_a'].map(len).describe()

In [ ]:
# 对测试集的长度统计分析，可以看出在长度上分布与训练数据相近
test['text_a'].map(len).describe()

In [ ]:
#待研究!
# 全局设置解决matplotlib中文显示错误问题，参考：https://aistudio.baidu.com/aistudio/projectdetail/1658980
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

import matplotlib.font_manager as font_manager

# 设置显示中文
matplotlib.rcParams['font.sans-serif'] = ['FZSongYi-Z13S'] # 指定默认字体
matplotlib.rcParams['axes.unicode_minus'] = False # 解决保存图像是负号'-'显示为方块的问题
# 设置字体大小
matplotlib.rcParams['font.size'] = 16

# 可视化类别标签分布情况
train['label'].value_counts(normalize=True).plot(kind='bar');

#X
#尚無停用詞(檔)
with open('stopwords.txt','r',encoding='utf-8') as f:
    stopwords = f.readlines()
stopwords = [i.strip() for i in stopwords]

#X
#结巴分词并去除停用词
def jieba_stop(each):
    result = []
    jieba_line = jieba.lcut(each)  #jieb.cut()返回迭代器，节省空间；jieba.lcut() 直接返回分词后的列表。三种分词模式：默认精确模式，用于文本分析  
    for l in jieba_line:
        if (l not in stopwords) and (l != ''): # 去除停用词以及空格
            result.append(l.strip())
    return result
xx = train.apply(jieba_stop)  #注意 apply 的用法
# date_news.news = date_news.news.apply(lambda x: ','.join([i for i in x if i !=''])) #反复去除空字符

##分詞(jieba) 目前僅 tain有分詞.test ,dev尚無
未來可用停用詞

In [ ]:
#使用jieba
# import jieba
# for index, row in train.iterrows():
#     # 對 row['text_a'] 進行分詞
#     words = jieba.cut(row['text_a'])
#     # 将分詞结果转换为字符串，并存回 train 變量中
#     train.loc[index, 'text_a'] = ','.join(words)
#============================================================
#使用ckip_client
# import ckip_client
# for index, row in train.iterrows():
#   # 對 row['text_a'] 進行分詞
#   words = ckip_client.segment(row['text_a'])
#   # 将分詞结果转换为字符串，并存回 train 變量中
#   train.loc[index, 'text_a'] = ','.join(words)
#=======================================================
# 直接使用主題模型提取特徵字
# 導入所需模塊
from gensim import corpora, models

# 分類的label
label_list = ['彩票', '財經', '遊戲', '家居', '房產', '星座', '體育', '股票', '時政', '科技', '娛樂', '社會', '教育', '時尚']

# 分類的文本
text_list = train['text_a'].tolist()

# 定義停用詞
original_stop_words = ["，", "。", "；", "：", "“", "”", "（", "）", "《", "》", "、", "？", "「", "」", "『", "』", "‘", "’", "...", "0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]

with open("/content/drive/MyDrive/中文新闻文本标题分类Data/stop_words.txt", "r", encoding="utf-8") as f:
  stop_words = f.read().split("\n")
stop_words += original_stop_words
# print(stop_words)
# # 載入停用詞至jieba
# for word in stop_words:
#     jieba.add_word(word)
#斷詞
# texts = [[word for word in jieba.cut(document)] for document in text_list]
texts = [[word for word in jieba.cut(document) if word not in stop_words] for document in text_list]

# 建立詞袋字典
dictionary = corpora.Dictionary(texts)

# 建立詞袋資料集
corpus = [dictionary.doc2bow(text) for text in texts]

# 建立LDA模型
lda = models.LdaModel(corpus=corpus, id2word=dictionary, num_topics=len(label_list))

# 使用下面的方式，指定每個主題的名稱
for i, topic in enumerate(lda.show_topics(num_topics=len(label_list), num_words=10)):
  print('主題{}:{}'.format(label_list[i], topic))

# 註: 上述程式碼中的num_topics參數可以根據你的需求調整，num_words參數可以決定每個主題中顯示的詞語數量。

In [ ]:
texts

In [ ]:
train

##词频统计

In [ ]:
row['text_a'].split(',')#row['text_a'].split('')兩者結果一樣

In [ ]:
import collections
# 定義一個空的列表
word_counts_list = []#一定要定義，不定義會NameError: name 'word_counts_list' is not defined

# 1.遍歷 train 變量的每一行，然後對每一行的文本使用 
#   collections.Counter() 方法统计词频：
for index, row in train.iterrows():
  # 對 row['text_a'] 進行詞頻統計
  word_counts = collections.Counter(row['text_a'].split(','))
  # 2.将 collections.Counter() 返回的字典转换为一个序列（Series）：
  word_counts_series = pd.Series(word_counts)
  # 3. 将序列存到另外一個變量中：
  word_counts_list.append(word_counts_series)
# 4. 将 word_counts_list 中的所有序列合并成一個大的序列：
all_word_counts = pd.concat(word_counts_list)
# 5. 對 all_word_counts 進行排序，然後取出前 10 個最多的詞頻的詞：
# 對 all_word_counts 進行排序
sorted_word_counts = all_word_counts.sort_values(ascending=False)

# 取出前 10 個最多的詞頻的詞
top_10_words = sorted_word_counts[:10]


In [ ]:
word_counts

In [ ]:
import matplotlib.pyplot as plt

# 使用圖表的方式進行視覺化觀察
top_10_words.plot(kind='bar')
plt.show()

##TF-IDF
特徵選詞
TF拿高頻率詞
IDF去掉太高頻率詞(無用詞)e.g.的

In [ ]:
#TF-IDF 
tfidf = TfidfVectorizer()#選擇適合的公式
news_tfidf = tfidf.fit_transform(train['text_a'])#將公式套用至你的資料

In [ ]:
news_tfidf

In [ ]:
# 使用tfidf.get_feature_names()方法來檢視TF-IDF選擇的特徵詞。此方法返回一個列表，其中包含所有特徵詞。
feature_names = tfidf.get_feature_names()
for name in feature_names:
  print(name)
#other way
# # 使用tfidf.vocabulary_來得到每個詞在矩陣中的索引，這樣你就可以檢查每個詞在文本中出現的頻率
# vocab = tfidf.vocabulary_
# for word in vocab:
#   print(word, vocab[word])#vocab[word]是每個詞在字典中的索引值

In [ ]:
tfidf_matrix_array = news_tfidf.toarray()
print(tfidf_matrix_array)

In [ ]:
row['text_a']

In [ ]:
train

#再次用主題模型看看特徵詞選得好不好

In [ ]:
# 直接使用主題模型提取特徵字
# 導入所需模塊
from gensim import corpora, models

# 分類的label
label_list = ['彩票', '財經', '遊戲', '家居', '房產', '星座', '體育', '股票', '時政', '科技', '娛樂', '社會', '教育', '時尚']

# 分類的文本變feature_names
# text_list = train['text_a'].tolist()
feature_names = tfidf.get_feature_names()

# 定義停用詞
original_stop_words = ["，", "。", "；", "：", "“", "”", "（", "）", "《", "》", "、", "？", "「", "」", "『", "』", "‘", "’", "...", "0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]

with open("/content/drive/MyDrive/中文新闻文本标题分类Data/stop_words.txt", "r", encoding="utf-8") as f:
  stop_words = f.read().split("\n")
stop_words += original_stop_words
# print(stop_words)
# # 載入停用詞至jieba
# for word in stop_words:
#     jieba.add_word(word)
#斷詞
# texts = [[word for word in jieba.cut(document)] for document in text_list]
texts = [[word for word in jieba.cut(document) if word not in stop_words] for document in feature_names]

# 建立詞袋字典
dictionary = corpora.Dictionary(texts)

# 建立詞袋資料集
corpus = [dictionary.doc2bow(text) for text in texts]

# 建立LDA模型
lda = models.LdaModel(corpus=corpus, id2word=dictionary, num_topics=len(label_list))

# 使用下面的方式，指定每個主題的名稱
for i, topic in enumerate(lda.show_topics(num_topics=len(label_list), num_words=10)):
  print('主題{}:{}'.format(label_list[i], topic))

# 註: 上述程式碼中的num_topics參數可以根據你的需求調整，num_words參數可以決定每個主題中顯示的詞語數量。

4. 网格搜索寻最优模型及最优参数

In [ ]:
#跑很久
train_size = 1000
#数据集划分
y = train['label']
X_train = news_tfidf[:train_size,:]  #训练集X
X_test = news_tfidf[train_size:,:]   #测试集X
y_train = y.iloc[:train_size,]  
y_test = y.iloc[train_size:,]  
lr = LogisticRegression()  #模型拟合
MNB = MultinomialNB()
lr_params = {'C':[0.1,0.5,1,10],'random_state':[0]}  #参数
MNB_params = {'alpha':[0.1,0.5,1,10]}

rf = RandomForestRegressor()
rf_params = {'n_estimators':[500,1000],'max_depth':[5,10],'random_state':[0]}

 #网格搜索
lr_grid = GridSearchCV(lr,lr_params,n_jobs=-1,cv=10,verbose=1,scoring='neg_mean_squared_error')
lr_grid.fit(X_train,y_train)
MNB_grid = GridSearchCV(MNB,MNB_params,n_jobs=-1,cv=10,verbose=1,scoring='neg_mean_squared_error')
MNB_grid.fit(X_train,y_train)

rf_grid = GridSearchCV(rf,rf_params,n_jobs=-1,cv=10,verbose=1,scoring='neg_mean_squared_error')
rf_grid.fit(X_train,y_train)

print('LR best params: '+ str(lr_grid.best_params_))  #最优参数
print('LR best score: '+ str(lr_grid.best_score_))        #最优分数

print('RF best params: '+ str(rf_grid.best_params_))
print('RF best score: '+ str(rf_grid.best_score_))

print('MNB best params: '+ str(MNB_grid.best_params_))
print('MNB best score: '+ str(MNB_grid.best_score_))

5.預測

In [ ]:
y_pred = lr_grid.predict(X_test) #使用 LogisticRegression 模型进行预测

y_pred_MNB = MNB_grid.predict(X_test) #使用 MultinomialNB 模型进行预测

#y_pred_rf = rf_grid.predict(X_test) #使用 RandomForestRegressor 模型进行预测

In [ ]:
y_pred


In [ ]:
y_pred = lr_grid.predict("測試分類跑出結果")

6.評估模型

In [ ]:
y_test

In [ ]:
type(y_test)

In [ ]:
type(y_pred_lr)

隨機森林

In [ ]:
# #精度（precision）
# from sklearn.metrics import precision_score
# y_pred_rf = rf_gri.predict(X_test)
# precision = precision_score(y_test, y_pred_rf, average='micro') # 计算 micro 召回率
# print("precision (micro): ", precision)

# precision = precision_score(y_test, y_pred_rf, average='macro') # 计算 macro 召回率
# print("precision (macro): ", precision)

# precision = precision_score(y_test, y_pred_rf, average='weighted') # 计算 weighted 召回率
# print("precision (weighted): ", precision)

MultinomialNB

In [ ]:
#精度（precision）
from sklearn.metrics import precision_score
y_pred_MNB = MNB_grid.predict(X_test)
precision = precision_score(y_test, y_pred_MNB, average='micro') # 计算 micro 召回率
print("precision (micro): ", precision)

precision = precision_score(y_test, y_pred_MNB, average='macro') # 计算 macro 召回率
print("precision (macro): ", precision)

precision = precision_score(y_test, y_pred_MNB, average='weighted') # 计算 weighted 召回率
print("precision (weighted): ", precision)

lr

In [ ]:
#精度（precision）
from sklearn.metrics import precision_score
y_pred_lr = lr_grid.predict(X_test)
precision = precision_score(y_test, y_pred_lr, average='micro') # 计算 micro 召回率
print("precision (micro): ", precision)

precision = precision_score(y_test, y_pred_lr, average='macro') # 计算 macro 召回率
print("precision (macro): ", precision)

precision = precision_score(y_test, y_pred_lr, average='weighted') # 计算 weighted 召回率
print("precision (weighted): ", precision)


In [ ]:
#召回率（recall）
from sklearn.metrics import recall_score

y_pred_lr = lr_grid.predict(X_test)
recall = recall_score(y_test, y_pred_lr, average='micro') # 计算 micro 召回率
print("Recall (micro): ", recall)

recall = recall_score(y_test, y_pred_lr, average='macro') # 计算 macro 召回率
print("Recall (macro): ", recall)

recall = recall_score(y_test, y_pred_lr, average='weighted') # 计算 weighted 召回率
print("Recall (weighted): ", recall)


##模型應用

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 定义新的数值特征
new_data = ['这是一篇关于机器学习的文章', '这是一篇关于人工智能的文章']

# 定义 TfidfVectorizer 实例
tfidf = TfidfVectorizer()

# 使用训练数据训练 TfidfVectorizer
tfidf.fit(train['text_a'])

# 使用训练好的 TfidfVectorizer 转换新的数值特征
new_tfidf = tfidf.transform(new_data)

# 使用训练好的模型 lr_grid 预测新的数值特征
predictions = lr_grid.predict(new_tfidf)

print(predictions)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 定义新的数值特征
new_data = ['''講座內容：
在講座中，王小棣導演將從自身經驗出發，分享豐富的生命故事與人生歷練，並帶出近期拍攝、上映之影集【茁劇場–誰說媽媽像月亮】，分享拍攝過程與內容，以及個人理念。講座第一個小時為 【茁劇場–誰說媽媽像月亮】之影片欣賞。

講師簡介：
王小棣導演於1979年回台工作，擔任多部影片導演與編劇，數度獲得金鐘獎肯定，近期著名作品為 《 45 度 C 天空下》、《 波麗士大人》、《 我在墾丁天氣晴》……等電視劇。
2016 年，王小棣導演結合多名導演，催生【植劇場】， 深耕臺灣戲劇，培植優秀戲劇從業人員。植劇場在第 52 屆電 視金鐘獎取得 24 項入圍紀錄，並奪得金鐘獎戲劇節目、戲劇節目女 主角、戲劇節目女配角……等 5 項大獎，獲得外界極大肯定。
2022年，開啟植劇場的續篇【茁劇場】，持續深耕台灣戲劇界。''']

# 定义 TfidfVectorizer 实例
tfidf = TfidfVectorizer()

# 使用训练数据训练 TfidfVectorizer
tfidf.fit(train['text_a'])

# 使用训练好的 TfidfVectorizer 转换新的数值特征
new_tfidf = tfidf.transform(new_data)

# 使用训练好的模型 lr_grid 预测新的数值特征
predictions = lr_grid.predict(new_tfidf)

print(predictions)

In [ ]:
new_tfidf

##舊版

from sklearn.feature_extraction.text import TfidfVectorizer

# 定义新的数值特征
new_samples = ['這是一條新聞', '這是一個很有意思的事情']

# 使用训练时使用的 TfidfVectorizer 转换为 Tf-idf 矩阵
# vectorizer = TfidfVectorizer(stop_words=stopwords)
vectorizer = TfidfVectorizer()
X_new = vectorizer.fit_transform(new_samples)

# 使用 lr_grid 预测分类结果
y_pred = lr_grid.predict(X_new)

print(y_pred)  # 输出 ['科技', '娱乐']

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
<ipython-input-48-6ac667864f40> in <module>
     10 
     11 # 使用 lr_grid 预测分类结果
---> 12 y_pred = lr_grid.predict(X_new)
     13 
     14 print(y_pred)  # 输出 ['科技', '娱乐']

5 frames
/usr/local/lib/python3.8/dist-packages/sklearn/base.py in _check_n_features(self, X, reset)
    398 
    399         if n_features != self.n_features_in_:
--> 400             raise ValueError(
    401                 f"X has {n_features} features, but {self.__class__.__name__} "
    402                 f"is expecting {self.n_features_in_} features as input."

ValueError: X has 2 features, but LogisticRegression is expecting 45190 features as input.

In [ ]:
X_train = news_tfidf[:train_size,:]
X_train

In [ ]:
MNB_grid.fit(X_train, y_train)
y_predict = MNB_grid.predict(X_test)     #预测值
mat = confusion_matrix(y_test,y_predict) #混淆矩阵


In [ ]:
mat

In [ ]:
df = pd.DataFrame(mat,columns=label_list,index=label_list)
report_MNB = classification_report(y_test,y_predict)

In [ ]:
df